In [ ]:
import sys
from pathlib import Path

# find the src/ folder with data_loader.py (same setup as inspecting.ipynb)
here = Path.cwd()
src = next((p / "src" for p in [here, *here.parents] if (p / "src" / "data_loader.py").exists()), None)
assert src is not None, "data_loader.py not found: put this notebook in the notebooks/ folder"
sys.path.insert(0, str(src))
from data_loader import INPUT_DIR, load_long, load_variables

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, clone
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

N_COMPONENTS = 15                 # PCA components kept for T²
POSITIVE = "accepted"             # class the leaderboard treats as positive ("accepted" or "rejected")
BORDERLINE = ["A_101", "A_102", "A_103", "A_104"]   # brief: rejected, just outside acceptability
PROCESSES = ["A", "B"]

In [ ]:
class PCAMonitor(BaseEstimator):
    """Multivariate statistical process control: Hotelling's T² and/or Q (SPE) residual.
    score_samples returns higher = more normal (scikit-learn convention)."""

    def __init__(self, n_components=10, stat="T2"):
        self.n_components = n_components
        self.stat = stat

    def _raw(self, X):
        X = np.asarray(X)
        T = self.pca_.transform(X)
        t2 = (T ** 2 / self.pca_.explained_variance_).sum(axis=1)
        q = ((X - self.pca_.inverse_transform(T)) ** 2).sum(axis=1)
        return t2, q

    def fit(self, X, y=None):
        self.pca_ = PCA(self.n_components).fit(np.asarray(X))
        t2, q = self._raw(X)
        self.t2_med_, self.q_med_ = np.median(t2), np.median(q)
        return self

    def score_samples(self, X):
        t2, q = self._raw(X)
        t2, q = t2 / self.t2_med_, q / self.q_med_
        return -{"T2": t2, "Q": q, "combined": t2 + q}[self.stat]


def build_model():
    return make_pipeline(StandardScaler(), PCAMonitor(N_COMPONENTS, "T2"))

In [ ]:
# ---------------------------------------------------------------- model
class PCAMonitor(BaseEstimator):
    """Multivariate statistical process control: Hotelling's T² and/or Q (SPE) residual.
    score_samples returns higher = more normal (scikit-learn convention)."""

    def __init__(self, n_components=10, stat="T2"):
        self.n_components = n_components
        self.stat = stat

    def _raw(self, X):
        X = np.asarray(X)
        T = self.pca_.transform(X)
        t2 = (T ** 2 / self.pca_.explained_variance_).sum(axis=1)
        q = ((X - self.pca_.inverse_transform(T)) ** 2).sum(axis=1)
        return t2, q

    def fit(self, X, y=None):
        self.pca_ = PCA(self.n_components).fit(np.asarray(X))
        t2, q = self._raw(X)
        self.t2_med_, self.q_med_ = np.median(t2), np.median(q)
        return self

    def score_samples(self, X):
        t2, q = self._raw(X)
        t2, q = t2 / self.t2_med_, q / self.q_med_
        return -{"T2": t2, "Q": q, "combined": t2 + q}[self.stat]


def build_model():
    return make_pipeline(StandardScaler(), PCAMonitor(N_COMPONENTS, "T2"))

In [ ]:
def base_features(d, features):
    """Whole-batch mean/std/min/max and linear slope over time, per sensor."""
    g = d.groupby("batch_id")
    bf = g[features].agg(["mean", "std", "min", "max"])
    bf.columns = ["_".join(c) for c in bf.columns]

    t_c = d["t"] - g["t"].transform("mean")
    x_c = d[features] - g[features].transform("mean")
    slopes = x_c.mul(t_c, axis=0).groupby(d["batch_id"]).sum().div(
        (t_c ** 2).groupby(d["batch_id"]).sum(), axis=0)
    slopes.columns = [f"{f}_slope" for f in features]
    return bf.join(slopes)


def extra_features(d, features):
    """Phase means, robust range, roughness, local instability, sensor-pair correlations."""
    g = d.groupby("batch_id")

    t_norm = d["t"] / g["t"].transform("max")
    phase = pd.cut(t_norm, [-0.01, 0.2, 0.8, 1.0], labels=["start", "mid", "end"])
    ph = d.groupby(["batch_id", phase], observed=True)[features].mean().unstack()
    ph.columns = [f"{f}_{p}" for f, p in ph.columns]

    q = g[features].quantile([0.05, 0.95]).unstack()
    q.columns = [f"{f}_q{int(p * 100)}" for f, p in q.columns]

    same = d["batch_id"].eq(d["batch_id"].shift())
    rough = d[features].diff().where(same).abs().groupby(d["batch_id"]).mean()
    rough.columns = [f"{f}_rough" for f in features]

    vol = g[features].rolling(50).std().groupby(level=0).max()
    vol.columns = [f"{f}_maxvol" for f in features]

    c = g[features].corr().unstack()
    pairs = [(a, b) for i, a in enumerate(features) for b in features[i + 1:]]
    corr = c[pairs].fillna(0)
    corr.columns = [f"corr_{a}__{b}" for a, b in pairs]

    return ph.join(q).join(rough).join(vol).join(corr)


def make_features(d, features):
    return base_features(d, features).join(extra_features(d, features))

In [ ]:
def load_labels(process):
    labels = pd.read_csv(INPUT_DIR / f"process {process}/calibration data/calibration_labels.csv")
    labels.loc[labels["batch_id"].isin(BORDERLINE), "label"] = "rejected"   # follow the brief
    return labels.set_index("batch_id")["label"]


def loo_scores(model, X, acc):
    """Leave-one-accepted-batch-out. Returns anomaly scores (higher = more anomalous)."""
    X_acc, X_rej = X[acc], X[~acc]
    acc_s = pd.Series(index=X_acc.index, dtype=float)
    rej_s = []
    for b in X_acc.index:
        m = clone(model).fit(X_acc.drop(index=b))
        acc_s[b] = -m.score_samples(X_acc.loc[[b]])[0]
        rej_s.append(-m.score_samples(X_rej))
    return acc_s, pd.Series(np.mean(rej_s, axis=0), index=X_rej.index)


def choose_threshold(acc_s, rej_s, positive=POSITIVE):
    """Threshold on the anomaly score that maximises F1 (score > threshold -> rejected)."""
    s = pd.concat([acc_s, rej_s])
    is_acc = s.index.isin(acc_s.index)
    y = is_acc.astype(int) if positive == "accepted" else (~is_acc).astype(int)
    best_f1, best_t = -1.0, None
    for t in np.unique(s):
        pred = (s <= t).astype(int) if positive == "accepted" else (s > t).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

In [ ]:
def run(process):
    features = load_variables(process)

    cal = load_long(process, "calibration")
    test = load_long(process, "test")
    labels = load_labels(process)

    X = make_features(cal, features)
    acc = (labels.loc[X.index] == "accepted").to_numpy()
    X_test = make_features(test, features)[X.columns]

    model = build_model()
    acc_s, rej_s = loo_scores(model, X, acc)
    thr, cal_f1 = choose_threshold(acc_s, rej_s)

    final = clone(model).fit(X[acc])
    test_scores = pd.Series(-final.score_samples(X_test), index=X_test.index)

    submission = pd.DataFrame({
        "batch_id": test_scores.index,
        "label": np.where(test_scores > thr, "rejected", "accepted"),
    })
    submission.to_csv(f"submission_{process}.csv", index=False)

    print(f"process {process}: {X.shape[0]} calibration batches ({acc.sum()} accepted), "
          f"{X.shape[1]} features, {len(X_test)} test batches")
    print(f"threshold {thr:.4f} | leave-one-batch-out F1 {cal_f1:.3f} (positive = {POSITIVE})")
    print(f"flagged: {(acc_s > thr).mean():.0%} of calibration accepted, "
          f"{(rej_s > thr).mean():.0%} of calibration rejected, "
          f"{(test_scores > thr).mean():.0%} of test")
    print(f"saved submission_{process}.csv\n")
    return submission

In [ ]:
submissions = {p: run(p) for p in PROCESSES}

In [ ]:
submissions["A"].head()

In [ ]:
submission_all = pd.concat([submissions["A"], submissions["B"]], ignore_index=True)
submission_all.to_csv("submission.csv", index=False)

print(submission_all.shape)
print(submission_all["batch_id"].str[0].value_counts())   # batches per process
submission_all.head()

In [ ]:
import numpy as np
import pandas as pd
import requests

In [ ]:
BASE_URL = "https://leaderboard-project2-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "submission.csv"

def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


# def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    # submission = {"predictions": df.to_dict(orient="records")}
    # response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    # if (response.status_code // 100) != 2:
        # raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    # return response.json()["metrics"]

In [ ]:
get_leaderboard()

In [ ]:
# template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
# Change TEMPLATE with your team name as "Team XX", for example "Team 21" 
# metrics = submit(template, user="Team 13")
# metrics